In [10]:
import findspark
findspark.init("/opt/homebrew/opt/apache-spark/libexec")

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator, BinaryClassificationEvaluator

spark = SparkSession.builder \
    .appName("US_Accidents_ML") \
    .master("spark://localhost:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .config("spark.default.parallelism", "10") \
    .config("spark.executor.cores", "2") \
    .config("spark.memory.fraction", "0.8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Version:", spark.version)
print("Spark ML session started")

Spark Version: 4.1.1
Spark ML session started


In [ ]:
# Loading ML ready data from local parquet

LOCAL_ML = "file:///Users/sanjaybharvad/us_accidents_project/ml_ready_local/ml_ready"

df = spark.read.parquet(LOCAL_ML)

# Convert boolean road columns to integer
road_features = [
    "Amenity", "Crossing", "Junction", "Stop",
    "Traffic_Signal", "Roundabout", "Bump", "Railway"
]
for c in road_features:
    df = df.withColumn(c, col(c).cast(IntegerType()))

# Define features
numeric_features = [
    "Start_Lat", "Start_Lng", "Distance(mi)",
    "Temperature(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)",
    "Hour", "Month", "Year", "DayOfWeek",
    "Is_RushHour", "Is_Weekend"
]

all_features = numeric_features + road_features
df = df.fillna(0, subset=all_features)
df = df.fillna(0, subset=["Duration_Min"])

print("Data loaded from local parquet")
print("Total features:", len(all_features))

[Stage 0:>                  (0 + 0) / 1][Stage 1:>                  (0 + 0) / 1]